# MDL on dependency-tree-style graphs

This cleaned notebook preserves the syntax experiment as an explicit negative-
result workflow. It uses handcrafted dependency structures by default, so it
runs without a language model download. Replace `documents` with graphs from a
spaCy or Stanza adapter for the full experiment.

In [ ]:
import math
import networkx as nx

from buhito.mdl import MDLGraphCompressor, labeled_isomorphic

In [ ]:
def dependency_graph(tokens, heads, pos, relations):
    graph = nx.Graph()
    for index, token in enumerate(tokens):
        graph.add_node(index, token=token, pos=pos[index])
    for child, head in enumerate(heads):
        if child == head:
            continue
        graph.add_edge(child, head, dep=relations[child])
    return graph


documents = [
    dependency_graph(
        ["models", "compress", "graphs"],
        [1, 1, 1],
        ["NOUN", "VERB", "NOUN"],
        ["nsubj", "ROOT", "obj"],
    ),
    dependency_graph(
        ["agents", "learn", "policies", "quickly"],
        [1, 1, 1, 1],
        ["NOUN", "VERB", "NOUN", "ADV"],
        ["nsubj", "ROOT", "obj", "advmod"],
    ),
    dependency_graph(
        ["the", "graph", "contains", "motifs"],
        [1, 2, 2, 2],
        ["DET", "NOUN", "VERB", "NOUN"],
        ["det", "nsubj", "ROOT", "obj"],
    ),
    dependency_graph(
        ["compression", "may", "not", "help"],
        [3, 3, 3, 3],
        ["NOUN", "AUX", "PART", "VERB"],
        ["nsubj", "aux", "neg", "ROOT"],
    ),
] * 4

len(documents)

In [ ]:
train_graphs = documents[:12]
test_graphs = documents[12:]

compressor = MDLGraphCompressor(
    graphlet_sizes=(3,),
    n_rules=5,
    min_graph_support=2,
    min_occurrences=4,
    max_candidates=25,
    node_label_keys="pos",
    edge_label_keys="dep",
    selector="sparse",
    dictionary_selection="best",
    cache_dir="artifacts/notebook_cache/syntax",
    validate=True,
    progress=True,
)
compressor.fit(train_graphs)
result = compressor.transform(test_graphs)

result.report

In [ ]:
display(compressor.candidate_table_)
display(compressor.dictionary_path_)
display(result.per_graph)

In [ ]:
decoded = result.decoded_graphs()
assert all(
    labeled_isomorphic(original, reconstructed)
    for original, reconstructed in zip(test_graphs, decoded)
)
print("Exact selected-label decoding verified.")

## Negative-result interpretation

The earlier syntax experiments found that small treelet features were expensive
and did not outperform simpler dependency-relation n-grams. An empty MDL
dictionary or negative savings is therefore an expected scientific outcome,
not a software failure. This notebook is the reproducible package-backed entry
point for extending that analysis.